In [28]:
%load_ext autoreload
%autoreload 2

import os
# Nạp biến môi trường (nếu file config.py của bạn chưa tự nạp)
from dotenv import load_dotenv
load_dotenv()

print("Môi trường test đã sẵn sàng!")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Môi trường test đã sẵn sàng!


In [29]:
# Import class từ file của bạn
from clients import FPTAIClient
from utils import JobMatcher
from schemas import TextRequest, MatchRequest
from config import settings

# Khởi tạo client
ai_client = FPTAIClient()
matcher = JobMatcher()

print(f"Đang dùng API tại: {settings.ENDPOINT}")

Đang dùng API tại: https://mkp-api.fptcloud.com


In [30]:
# Test 1: Refine CV
raw_text = "Mình tên Nam, làm kế toán 5 năm, biết Excel và Misa."
print("--- Đang gửi yêu cầu Refine ---")

try:
    refined_text = ai_client.chat_refine(raw_text)
    print("✅ KẾT QUẢ REFINE:")
    print(refined_text)
    for chunk in ai_client.chat_respond("Hãy recommend những công việc phù hợp nhất với tôi đi"):
        delta = chunk["choices"][0]["delta"].get("content", "")
        print(delta, end="", flush=True)
except Exception as e:
    print(f"❌ Lỗi: {e}")

--- Đang gửi yêu cầu Refine ---
✅ KẾT QUẢ REFINE:
Nam, 5 năm kinh nghiệm kế toán, thành thạo Excel và Misa.
<think>
Okay, let's see. The user is asking me to recommend suitable jobs for them. They mentioned they're an HR assistant. First, I need to figure out what their skills and experiences are. As an HR assistant, they probably have experience in recruitment, employee relations, administrative tasks, maybe payroll or benefits administration. They might also be familiar with HR software like SAP or Workday.

But wait, the user didn't provide specific details about their interests or what they want in a job. Should I ask more questions? The original request just says "recommend những công việc phù hợp nhất với tôi đi" which translates to "recommend the most suitable jobs for me." Since they're an HR assistant, maybe they're looking to advance in HR or transition into a different role. I should consider common career paths for HR assistants.

Possible roles could be HR Generalist, HR M

In [15]:
# Test 2: Embedding
print("--- Đang tạo Embedding ---")
vec_cv = ai_client.get_embedding(refined_text) # Dùng kết quả cell trên
vec_job = ai_client.get_embedding("Tuyển kế toán trưởng kinh nghiệm 5 năm, thành thạo Misa")

print(f"✅ Vector CV (10 số đầu): {vec_cv[:10]}")
print(f"Kích thước vector: {len(vec_cv)}")

# Test 3: Matching
score = matcher.compute_similarity(vec_cv, vec_job)
print(f"\n🎯 ĐỘ PHÙ HỢP: {score:.4f}")

--- Đang tạo Embedding ---
✅ Vector CV (10 số đầu): [-0.0479731522500515, 0.0486140139400959, 0.00862875860184431, -0.016357257962226868, 0.019912520423531532, 0.013862471096217632, 0.003042190568521619, 0.05529729649424553, -0.003150908276438713, -0.05227608606219292]
Kích thước vector: 1024

🎯 ĐỘ PHÙ HỢP: 0.2015
